In [ ]:
!pip install unbabel-comet
!pip uninstall -y tensorflow tensorboard

In [ ]:
from comet import download_model, load_from_checkpoint


def load_lines(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f.readlines()]


src = load_lines("test_en.txt")
ref = load_lines("test_it.txt")
sys_baseline = load_lines("baseline_qwen4b.txt")
sys_finetuned = load_lines("post_tuning_qwen_4b_optm.txt")


data_baseline = [{"src": s, "mt": m, "ref": r} for s, m, r in zip(src, sys_baseline, ref)]
data_finetuned = [{"src": s, "mt": m, "ref": r} for s, m, r in zip(src, sys_finetuned, ref)]

model_path = download_model("Unbabel/wmt22-comet-da")
model = load_from_checkpoint(model_path)


ì
baseline_output = model.predict(data_baseline, batch_size=8, gpus=1)

finetuned_output = model.predict(data_finetuned, batch_size=8, gpus=1)


score_base = baseline_output.system_score
score_fine = finetuned_output.system_score
delta = score_fine - score_base

print(f"\n" + "="*40)
print(f"Prestazioni (COMET SCORE)")
print(f"="*40)
print(f"Punteggio baseline (Zero-Shot): {score_base:.4f}")
print(f"Punteggio post-tuning (LoRA):   {score_fine:.4f}")
print(f"-"*40)
print(f"Miglioramento (Delta COMET):    {delta:+.4f}")
print(f"="*40)

Download dell'architettura neurale di valutazione (wmt22-comet-da)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...


Avvio dell'inferenza valutativa...
Calcolo COMET per la Baseline in corso...


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 17/17 [00:02<00:00,  6.09it/s]
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automa

Calcolo COMET per il Modello Post-Tuning in corso...


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 17/17 [00:02<00:00,  7.30it/s]



ANALISI DELLE PRESTAZIONI (COMET SCORE)
Punteggio Baseline (Zero-Shot): 0.6001
Punteggio Post-Tuning (LoRA):   0.4196
----------------------------------------
Miglioramento (Delta COMET):    -0.1805


In [ ]:

!pip install sacrebleu

import sacrebleu


def load_lines(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f.readlines()]


ref = load_lines("test_it.txt")
sys_baseline = load_lines("baseline_qwen4b.txt")
sys_finetuned = load_lines("post_tuning_qwen_4b_optm.txt")


bleu_base = sacrebleu.corpus_bleu(sys_baseline, [ref])
bleu_fine = sacrebleu.corpus_bleu(sys_finetuned, [ref])
delta_bleu = bleu_fine.score - bleu_base.score


chrf_base = sacrebleu.corpus_chrf(sys_baseline, [ref])
chrf_fine = sacrebleu.corpus_chrf(sys_finetuned, [ref])
delta_chrf = chrf_fine.score - chrf_base.score

print("="*50)
print("Prestazioni (BLEU & chrF SCORES)")
print("="*50)
print(f"BLEU:")
print(f"  Baseline:     {bleu_base.score:.2f}")
print(f"  Post-tuning:  {bleu_fine.score:.2f}")
print(f"  Delta:        {delta_bleu:+.2f}")
print("-"  * 50)
print(f"chrF")
print(f"  Baseline:     {chrf_base.score:.2f}")
print(f"  Post-tuning:  {chrf_fine.score:.2f}")
print(f"  Delta:        {delta_chrf:+.2f}")
print("="*50)

Avvio del calcolo delle metriche lessicali e di carattere...

ANALISI DELLE PRESTAZIONI (BLEU & chrF SCORES)
METRICA BLEU (Sovrapposizione di parole):
  Baseline:     12.18
  Post-Tuning:  0.18
  Delta:        -12.00
--------------------------------------------------
METRICA chrF (Sovrapposizione di caratteri):
  Baseline:     29.26
  Post-Tuning:  12.24
  Delta:        -17.03
